# Ñ y C1EÑ desde Python, en las T4 de Kaggle

Este cuaderno enseña cómo usar Ñ como una biblioteca de Python: `c1en.correr` para programas enteros y `c1en.funcion` para llamar funciones de Ñ con arreglos de NumPy, en la GPU cuando hay.

**Antes de correr:** Settings → Accelerator → **GPU T4 x2**, e Internet **On** (para clonar el repositorio).

In [ ]:
%%bash
set -e
if ! command -v node >/dev/null || [ "$(node -e 'console.log(+process.versions.node.split(".")[0])')" -lt 18 ]; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1 && apt-get install -y nodejs >/dev/null 2>&1
fi
node -v
if [ ! -d /kaggle/working/n ]; then git clone --depth 1 https://github.com/diegovelandiabarajas1-lang/Espanol-Like /kaggle/working/n; fi
cd /kaggle/working/n && npm install --omit=dev --no-audit --no-fund koffi >/dev/null 2>&1 && node motor/util/construir.js
nvidia-smi -L

In [ ]:
import sys, os
os.environ['N_RAIZ'] = '/kaggle/working/n'
sys.path.insert(0, '/kaggle/working/n/c1en')
import c1en, numpy as np
print('dispositivos que ve C1EÑ:', c1en.dispositivos())

## 1. Un programa de Ñ, tal cual

En el perfil C1EÑ no hace falta ningún `usar`: cálculo, fórmulas, tensores y `paralelo` están abiertos.

In [ ]:
print(c1en.correr('''
fijo f = "x^3 - 6x^2 + 11x - 6"
imprimir("f'(x) = ", derivarFormula(f, "x"))
imprimir("raíces: ", resolverFormula(f + " = 0", "x"))
imprimir("∫₀^∞ e^(-x²) dx = ", integrar(funcion(x) { devolver exp(-x * x) }, 0, |INFINITO))
fijo A = arreglo([[2, 1], [1, 3]], "real")
imprimir("autovalores: ", autovaloresGeneral(A))
imprimir("dispositivo: ", dispositivo())
'''))

## 2. Una función de Ñ llamada desde Python, con NumPy

`paralelo=True` la ejecuta una vez por índice, en la T4. Los arreglos de NumPy entran por referencia y vuelven modificados.

In [ ]:
tirar = c1en.funcion('''
fn tirar(i, dentro) {
  fijo x = azarHilo(7, i, 0)
  fijo y = azarHilo(7, i, 1)
  si x * x + y * y < 1.0 { sumarAtomico(dentro, 0, 1) }
}
''', paralelo=True)

import time
for N in (1_000_000, 20_000_000):
    dentro = np.zeros(1, dtype=np.int32)
    t0 = time.perf_counter()
    tirar(N, dentro)
    print(f'N={N:>11,}  π ≈ {4 * dentro[0] / N:.6f}  en {tirar.ultimo_dispositivo}  ({(time.perf_counter() - t0) * 1000:.0f} ms con arranque de Node)')

El mismo núcleo, forzando cada dispositivo. El valor de π tiene que ser **idéntico** en los tres: el azar es por contador, no por estado.

In [ ]:
for d in ('jit', 'nativo', 'gpu'):
    f = c1en.funcion(tirar.fuente, paralelo=True, dispositivo=d)
    dentro = np.zeros(1, dtype=np.int32)
    t0 = time.perf_counter()
    f(5_000_000, dentro)
    print(f'{d:>7}: {4 * dentro[0] / 5_000_000:.8f}  {(time.perf_counter() - t0) * 1000:6.0f} ms  ({f.ultimo_dispositivo})')

## 3. Un arreglo de NumPy entra, se transforma en la GPU y vuelve

Mandelbrot sobre una rejilla de 1600×1200: Ñ escribe en el arreglo de NumPy y matplotlib lo pinta.

In [ ]:
mandel = c1en.funcion('''
fn mandel(i, ancho, alto, iter, r) {
  var cx = -2.0 + 3.0 * (i % ancho) / ancho
  var cy = -1.2 + 2.4 * (i // ancho) / alto
  var zx = 0.0
  var zy = 0.0
  var k = 0
  mientras k < iter y zx * zx + zy * zy < 4.0 {
    var t = zx * zx - zy * zy + cx
    zy = 2.0 * zx * zy + cy
    zx = t
    k = k + 1
  }
  r[i] = k
}
''', paralelo=True)

ancho, alto = 1600, 1200
r = np.zeros(ancho * alto, dtype=np.int32)
t0 = time.perf_counter()
mandel(ancho * alto, ancho, alto, 500, r)
print(f'{ancho}×{alto}×500 en {mandel.ultimo_dispositivo}: {(time.perf_counter() - t0) * 1000:.0f} ms (con arranque y copia)')

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7.5)); plt.imshow(r.reshape(alto, ancho), cmap='magma', extent=(-2, 1, -1.2, 1.2)); plt.axis('off'); plt.show()

## 4. Las dos T4 a la vez

Un proceso por GPU, cada uno con su mitad de los índices. Como `azarHilo` depende solo del índice, la suma de las dos mitades es exactamente la cuenta de una sola GPU.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
N = 400_000_000
mitad = c1en.funcion('''
fn tirar(i, dentro, desde) {
  fijo j = i + desde
  fijo x = azarHilo(7, j, 0)
  fijo y = azarHilo(7, j, 1)
  si x * x + y * y < 1.0 { sumarAtomico(dentro, 0, 1) }
}
''', paralelo=True)

def en_gpu(k):
    f = c1en.funcion(mitad.fuente, paralelo=True, dispositivo='gpu', gpu=k)
    d = np.zeros(1, dtype=np.int32)
    f(N // 2, d, k * (N // 2))
    return int(d[0]), f.ultimo_dispositivo

t0 = time.perf_counter()
with ThreadPoolExecutor(2) as ex:
    (a, da), (b, db) = ex.map(en_gpu, (0, 1))
print(f'{da}: {a:,}   {db}: {b:,}')
print(f'π ≈ {4 * (a + b) / N:.8f} con {N:,} dardos en {time.perf_counter() - t0:.2f} s de pared')

## Cómo seguir

- Cualquier función de Ñ sirve con `c1en.funcion`, tenga o no `paralelo`; sin él, devuelve lo que la función devuelve (números, listas, diccionarios o arreglos).
- Dentro de un núcleo `paralelo` solo entran números, booleanos, arreglos y bucles; lo demás se rechaza con la línea.
- Los arreglos reales viajan como `float64` y los enteros como `int32`, que son los tipos de Ñ. Un arreglo de NumPy de otro tipo se convierte al entrar y se vuelve a convertir al salir.